# Lab 01 — PyTorch fundamentals & the two equations of backprop

**Goal:** verify experimentally that for a linear layer `Y = XW`:
- the **weight gradient** is `dL/dW = X.T @ delta` (needs the stored input activation → stays local in pipeline parallelism)
- the **activation gradient** is `dL/dX = delta @ W.T` (flows backward layer-to-layer → crosses the network in PP)

These two equations justify almost every design decision in the research proposal.

**Run on:** Google Colab, `Runtime → Change runtime type → T4 GPU` (CPU also works for this lab).

In [ ]:
import torch
print(torch.__version__)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, torch.cuda.get_device_name(0) if device=="cuda" else "")

## 1. Tensors and shapes
Everything in this field is a tensor with a shape. Discipline rule for all labs: after creating a tensor, print `.shape` and `.dtype`.

In [ ]:
x = torch.randn(2, 4, 8)          # think: Batch=2, Seq=4, Hidden=8
print(x.shape, x.dtype)
print("elements:", x.numel(), "bytes in fp32:", x.numel()*4, "bytes in fp16:", x.numel()*2)
x16 = x.half()
print(x16.dtype, "-> half the bytes, half the wire time")

## 2. Autograd in one picture
`requires_grad=True` tells PyTorch to build a computation graph. `loss.backward()` walks the graph in reverse (reverse-mode autodiff = the chain rule, mechanized) and fills `.grad` on every leaf.

In [ ]:
a = torch.tensor(3.0, requires_grad=True)
L = (a**2 + 2*a)          # dL/da = 2a + 2 = 8 at a=3
L.backward()
print(a.grad)             # expect tensor(8.)

## 3. The two equations — verified
We compute `delta = dL/dY` by hand for a simple loss `L = sum(Y**2)` (so `delta = 2Y`) and check PyTorch's grads against the formulas.

In [ ]:
torch.manual_seed(0)
X = torch.randn(4, 8, requires_grad=True)   # 'activation' entering the layer
W = torch.randn(8, 3, requires_grad=True)   # the layer's weight matrix
Y = X @ W                                    # forward
L = (Y**2).sum()
L.backward()

delta = 2*Y                                  # dL/dY by hand
print("weight grad   = X.T @ delta :", torch.allclose(W.grad, X.T @ delta, atol=1e-5))
print("activation grad = delta @ W.T:", torch.allclose(X.grad, delta @ W.T, atol=1e-5))
print("X.grad shape == X shape:", X.grad.shape == X.shape)

Three conclusions, memorize them:
1. `W.grad` needed `X` → **activations must be stored** during forward (this is the activation-memory problem; activation *checkpointing* = throw X away and recompute it later).
2. `X.grad` has the **same shape as X** → in pipeline parallelism the backward wire traffic equals the forward wire traffic.
3. `W.grad` is computed **inside** the layer → weight gradients never cross the network in PP. Only activation gradients do.

## 4. Memory accounting — the 16 bytes/param rule
Full training with Adam in mixed precision costs per parameter: 2 (fp16 weight) + 2 (fp16 grad) + 4 (fp32 master weight) + 4 (Adam m) + 4 (Adam v) = **16 bytes**.

In [ ]:
def training_memory_gb(n_params, full_finetune=True):
    if full_finetune:
        return n_params * 16 / 1e9
    # LoRA: frozen base = 2 bytes/param, adapters negligible at this scale
    return n_params * 2 / 1e9

for n, label in [(124e6, "GPT-2 124M"), (1.2e9, "Llama-3.2-1B"), (3.2e9, "Llama-3.2-3B"), (7e9, "7B")]:
    print(f"{label:14s} full: {training_memory_gb(n):7.1f} GB   LoRA base: {training_memory_gb(n, False):6.1f} GB")

The 7B row is why the proposal's workload is **LoRA fine-tuning**: 112 GB does not fit on 8–12 GB cards; ~14 GB split across 4 stages does.

## Exercises
1. Change the loss to `L = Y.abs().sum()` and recompute `delta` by hand (hint: `torch.sign`). Verify both equations still hold.
2. Chain two layers `Y = (X @ W1) @ W2` and verify `X.grad == (2*Y) @ W2.T @ W1.T` — the chain rule composing.
3. Add activations to the memory model: `B*S*H*2 bytes` per layer per micro-batch. For B=1, S=2048, H=4096, L=32: how many GB?